In [81]:
# ##### App2.py notebook SQL code for data handling and profiling using DuckDB
from tabnanny import check
import numpy as np
import pandas as pd
import duckdb
import re
import os

# =================================================================
# 1. SETUP
# =================================================================

# 1 Setup: Create a DuckDB database file in the same directory as your CSV
# 1.1 Load the actual CSV data first
df = pd.read_csv('SGJobData.csv') 

# 1.2 Connect using the new extension
con = duckdb.connect('SGJobData.duckdb')

# =================================================================
# 2. EXPLORATORY DATA ANALYSIS (EDA) & PROFILING
# =================================================================
# 2 Analysis: Use DuckDB to profile the data and identify issues
# 2.1 Summarize the DataFrame

# # SQL version
# summary = con.execute("SUMMARIZE SELECT * FROM df").df()
# display(summary)

# pandas version
# This replicates the DuckDB 'SUMMARIZE' table structure
# Updated pandas version that handles both numbers and text correctly
def get_summary(dataframe):
    # 1. Start with the basics (types, nulls, unique counts)
    summary = pd.DataFrame({
        'column_type': dataframe.dtypes,
        'nulls': dataframe.isna().sum(),
        'unique': dataframe.nunique()
    })
    
    # 2. Add min/max by checking the column type for each row
    # This prevents '9' being bigger than '10' for numbers
    summary['min'] = [dataframe[col].min() if dataframe[col].dtype != 'object' 
                      else dataframe[col].astype(str).min() for col in dataframe.columns]
    
    summary['max'] = [dataframe[col].max() if dataframe[col].dtype != 'object' 
                      else dataframe[col].astype(str).max() for col in dataframe.columns]
    
    return summary

# Use it on your filtered data
print("Initial Data Profile Summary")
display(get_summary(df))

# 2.2 Check for NULLs, Zeros, and empty strings in your key columns

# # SQL version of the null/zero/empty check
# missing_check = con.execute("""
#     SELECT 
#         SUM(CASE WHEN average_salary IS NULL THEN 1 ELSE 0 END) as null_salaries,
#         SUM(CASE WHEN average_salary = 0 THEN 1 ELSE 0 END) as zero_salaries,
#         SUM(CASE WHEN status_jobStatus IS NULL OR status_jobStatus = '' THEN 1 ELSE 0 END) as invalid_status
#     FROM df
# """).df()
# print("\nMissing and Invalid Data Check:")
# display(missing_check)

# pandas version of the above null/zero/empty check for quick reference
# This gives you a quick count of all nulls in every column
display(df[['average_salary', 'status_jobStatus']].isna().sum())

# To find the zeros specifically:
print("Zero Salaries:", (df['average_salary'] == 0).sum())

# 2.2 Salary Scale Outlier Detection
# Monthly salary of 12666400.0 could be an outlier. Further investigation is needed.

# # SQL version of the salary outlier check
# salary_outliers = con.execute("""
#     SELECT *
#     FROM df
#     WHERE average_salary > 1000000
# """).df()
# print("\nSalary Outliers:")
# display(salary_outliers)

# pandas version of the salary outlier check for quick reference
salary_outliers = df[df['average_salary'] > 1000000]

print("\nSalary Outliers:")
display(salary_outliers)

# 2.2.1 View the highest-paying roles to check for potential outliers or errors

# # SQL version of the top salary check

# top_salary_check = con.execute("""
#     SELECT 
#         title, 
#         postedCompany_name, 
#         average_salary, 
#         salary_type, 
#         positionLevels,
#         status_jobStatus
#     FROM df
#     WHERE average_salary > 0
#     ORDER BY average_salary DESC
#     LIMIT 10
# """).df()
# print("\nTop Salary Roles:")
# display(top_salary_check)

# pandas version of the top salary check for quick reference
top_salary_check = df[df['average_salary'] > 0][['title', 'postedCompany_name', 'average_salary', 'salary_type', 'positionLevels', 'status_jobStatus']].sort_values(by='average_salary', ascending=False).head(10)  
print("\nTop Salary Roles:")
display(top_salary_check)

# 2.2.2 See how many rows exist at different high-salary tiers

# # SQL version of the salary tiers check
# salary_tiers = con.execute("""
#     SELECT 
#         CASE 
#             WHEN average_salary > 1000000 THEN '1. Over $1M (Obvious Error)'
#             WHEN average_salary BETWEEN 100000 AND 1000000 THEN '2. $100k - $1M (Highly Suspicious)'
#             WHEN average_salary BETWEEN 50000 AND 100000 THEN '3. $50k - $100k (Top Tier / C-Suite)'
#             ELSE '4. Under $50k (Normal Market)'
#         END as monthly_salary_tier,
#         COUNT(*) as job_count
#     FROM df
#     WHERE salary_type = 'Monthly'
#     GROUP BY 1  /* Groups by monthly_salary_tier */
#     ORDER BY 1 /* Order by monthly_salary_tier */
# """).df()
# print("\nSalary Tiers:")
# display(salary_tiers)

# pandas version of the salary tiers check for quick reference
def categorize_salary(salary):
    if salary > 1000000:
        return '1. Over $1M (Obvious Error)'
    elif 100000 <= salary <= 1000000:
        return '2. $100k - $1M (Highly Suspicious)'
    elif 50000 <= salary < 100000:
        return '3. $50k - $100k (Top Tier / C-Suite)'
    else:
        return '4. Under $50k (Normal Market)'  
    
# Filter for Monthly and apply the function
monthly_df = df[df['salary_type'] == 'Monthly'].copy() # Create a copy to avoid SettingWithCopyWarning
monthly_df['monthly_salary_tier'] = monthly_df['average_salary'].apply(categorize_salary)

# Group by the new tier column and count
salary_tiers_pandas = monthly_df.groupby('monthly_salary_tier').size().reset_index(name='job_count').sort_values(by='monthly_salary_tier')
print("\nSalary Tiers (Pandas):")
display(salary_tiers_pandas)

# 2.2.3 Inspect the suspicious tiers to confirm they are errors

# # SQL version of the suspicious salary roles check
# sense_check = con.execute("""
#     SELECT 
#         title,
#         postedCompany_name,
#         average_salary,
#         positionLevels,
#         categories
#     FROM df
#     WHERE average_salary >= 100000
#     AND salary_type = 'Monthly'
#     ORDER BY average_salary DESC
# """).df()
# print("\nSuspicious Salary Roles:")
# display(sense_check.head(20))   

# pandas version of the suspicious salary roles check for quick reference
sense_check = df[(df['average_salary'] >= 100000) & (df['salary_type'] == 'Monthly')][['title', 'postedCompany_name', 'average_salary', 'positionLevels', 'categories']].sort_values(by='average_salary', ascending=False).head(20)
print("\nSuspicious Salary Roles:")
display(sense_check)

# 2.2.4 Check how many 'Juniors/Executives' are in the suspicious $100k+ group

# # SQL version of the suspicious group check
# suspicious_roles = con.execute("""
#     SELECT positionLevels, COUNT(*) as error_count, AVG(average_salary) as avg_val
#     FROM df
#     WHERE average_salary >= 100000
#     AND salary_type = 'Monthly'
#     GROUP BY 1
#     ORDER BY 2 DESC
# """).df()
# print("\nJuniors/Executives in Suspicious Group:")
# display(suspicious_roles)

# pandas version of the suspicious group check for quick reference
suspicious_roles = df[(df['average_salary'] >= 100000) & (df['salary_type'] == 'Monthly')].groupby('positionLevels').agg(error_count=('positionLevels', 'size'), avg_val=('average_salary', 'mean')).reset_index().sort_values(by='error_count', ascending=False)
print("\nJuniors/Executives in Suspicious Group:")
display(suspicious_roles)   

# 2.3 The "Executive" vs "Senior Management" Gap
# Run this to see EVERY possible label in the positionLevels column

# # SQL version of the distinct position levels check
# distinct_position_levels = con.execute("SELECT DISTINCT positionLevels FROM df").df()
# print("\n\n\nEVERY possible label in the positionLevels column")
# display(distinct_position_levels)

# pandas version of the distinct position levels check for quick reference
distinct_position_levels = df['positionLevels'].dropna().unique()
print("\n\n\nEVERY possible label in the positionLevels column")
display(distinct_position_levels)

# 2.3.1 Cross-checking seniority levels against real salary data

# # SQL version of the seniority check
# seniority_check = con.execute("""
#     SELECT 
#         positionLevels,     
#         COUNT(*) as job_count,
#         MIN(average_salary) as min_pay,
#         MAX(average_salary) as max_pay,
#         AVG(average_salary) as avg_pay
#     FROM df 
#     WHERE average_salary > 0 
#     AND average_salary < 100000 -- Keeping our 'Cleanliness Wall'
#     GROUP BY 1
#     ORDER BY avg_pay DESC
# """).df()

# print("\n\n\nCross-checking seniority levels against real salary data")
# display(seniority_check)

# pandas version of the seniority check for quick reference
seniority_check = df[(df['average_salary'] > 0) & (df['average_salary'] < 100000)].groupby('positionLevels').agg(job_count=('positionLevels', 'size'), min_pay=('average_salary', 'min'), max_pay=('average_salary', 'max'), avg_pay=('average_salary', 'mean')).sort_values(by='avg_pay', ascending=False).reset_index(drop=False)  
print("\n\n\nCross-checking seniority levels against real salary data")
display(seniority_check)

# 2.4 Missing Categories (The 0.38% Row Drop)
# Several of the columns (Categories, EmploymentTypes, Status) have the exact same null percentage (0.38%).
# Action: Instead of cleaning each column one by one, you can do a "bulk clean" by dropping any row where the title or postedCompany_name is missing.
# In a "Recruitment Intelligence" scenario, a row without a Title or a Company Name is completely useless, even if the other 0.38% columns (like Category or Status) were magically filled in.
# *****For Report: "I used 'title' and 'postedCompany_name' as the primary filters for row deletion 
# because they represent the 'minimum viable data' for a recruitment lead. Since these columns 
# shared a 0.38% null rate with other categorical fields, removing them effectively purged corrupted 
# rows without losing any actionable business intelligence."

# 2.5 Number_of_vacancies Max = 999
# The Observation: Your max vacancies is 999.
# The Weirdness: Often, "999" is a placeholder used by HR systems when they have "unlimited" or "many" roles.
# Business Impact: If you calculate Hiring Volume (Hiring 999 people @ $5k each), 
# it might artificially inflate the "ROI" of a company compared to a company hiring 1 person @ $100k

# 2.5.1 Instead: Calculate the 99th percentile directly in DuckDB
# This tells you what 99% of your job postings actually look like

# # SQL version of the 99th percentile check
# percentile_query = """
#     SELECT quantile(numberOfVacancies, 0.99) AS vacancy_limit
#     FROM df
#     WHERE numberOfVacancies > 0
# """
# vacancy_limit_df = con.execute(percentile_query).df()

# limit_result = vacancy_limit_df['vacancy_limit'][0]  #vacancy_limit_df is a DataFrame, we extract the value from the 'vacancy_limit' column
# print(f"The 99th percentile limit is: {limit_result}")

# pandas version of the 99th percentile check for quick reference
vacancy_limit = df[df['numberOfVacancies'] > 0]['numberOfVacancies'].quantile(0.99) #tried without numberofvacancies filter also 20 cap
print(f"The 99th percentile limit is: {vacancy_limit}")

# 2.5.2 Run this to see how many postings are stuck at that "999" ceiling:

# # SQL version of the 'stuck at 999' check
# stuck_at_999 = con.execute("""
#     SELECT
#         COUNT(*) as count_stuck_at_999
#     FROM df
#     WHERE numberOfVacancies = 999
# """).df()
# print("\n\n\nChecking how many postings are stuck at the '999' ceiling:")
# display(stuck_at_999)

# pandas version of the 'stuck at 999' check
# This counts how many rows have exactly 999 vacancies
stuck_at_999 = (df['numberOfVacancies'] == 999).sum()
print(f"\n\n\nChecking how many postings are stuck at the '999' ceiling: {stuck_at_999}")

#2.5.3 To verify if a cap of 20 is safe, you need to check if there are "High-Value Whales" 
#(Senior roles with high vacancy counts) that you might be accidentally clipping.

# # SQL version of the 'High-Value Whale Check'
# whale_check = con.execute("""
#     SELECT
#         positionLevels,
#         MAX(numberOfVacancies) as max_vacancies_found,
#         AVG(average_salary) as avg_salary,
#         COUNT(*) as total_postings
#     FROM df
#     WHERE average_salary > 1000 AND average_salary < 100000
#     GROUP BY 1
#     ORDER BY avg_salary DESC
# """).df()
# print("\n\n\nChecking if high-level roles actually have high vacancy counts:")
# display(whale_check)

# pandas version of the 'High-Value Whale Check' for quick reference
whale_check = df[(df['average_salary'] > 1000) & (df['average_salary'] < 100000)].groupby('positionLevels').agg(max_vacancies_found=('numberOfVacancies', 'max'), avg_salary=('average_salary', 'mean'), total_postings=('positionLevels', 'size')).sort_values(by='avg_salary', ascending=False).reset_index(drop=False)
print("\n\n\nChecking if high-level roles actually have high vacancy counts:")
display(whale_check)

# Results: Yes, there are some "Senior Management" roles with high vacancy counts (e.g., max_vacancies_found = 999), but these are likely data entry errors rather than genuine high-volume senior roles.
# The Impact: Capping the numberOfVacancies at 20 will prevent these outliers

# 2.5.4 See exactly how many rows are using these 'placeholder' numbers

# # SQL version of the 'Placeholder Check'
# placeholder_check = con.execute("""
#     SELECT numberOfVacancies, COUNT(*) as frequency
#     FROM df
#     WHERE numberOfVacancies > 20    
#     GROUP BY 1
#     ORDER BY 1 DESC
# """).df()
# print("\n\n\nChecking the frequency of high vacancy counts (potential placeholders):")
# display(placeholder_check)  

# pandas version of the 'Placeholder Check' for quick reference
placeholder_check = df[df['numberOfVacancies'] > 20].groupby('numberOfVacancies').size().reset_index(name='frequency').sort_values(by='numberOfVacancies', ascending=False)
print("\n\n\nChecking the frequency of high vacancy counts (potential placeholders):")
display(placeholder_check)

# Results: "The 'High-Value Whale Check' revealed that even senior roles with high average salaries do not have vacancy counts approaching the 999 placeholder.
# The 'Placeholder Check' confirmed that only 78 postings use the '999' value

# 2.5.5 Check the impact of the 'Cap at 20' rule

# # SQL version of the 'Cap at 20' impact check
# cap_impact = con.execute("""        
#     SELECT
#         CASE WHEN numberOfVacancies > 20 THEN 'Above Cap (Clipped to 20)'   
#                 ELSE 'At or Below Cap (Original Data)'
#         END as vacancy_status,
#         COUNT(*) as posting_count,
#         ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM df), 2) as percentage_of_total
#     FROM df
#     WHERE numberOfVacancies > 0
#     GROUP BY 1      
# """).df()

# print("\n\n\nChecking the impact of the 'Cap at 20' rule:")
# display(cap_impact)

# pandas version of the 'Cap at 20' impact check
# 1. Filter out zeros first (to match your SQL WHERE clause)
valid_vacancies = df[df['numberOfVacancies'] > 0]['numberOfVacancies']

# 2. Create a status label
status = (valid_vacancies > 20).map({
    True: 'Above Cap (Clipped to 20)', 
    False: 'At or Below Cap (Original Data)'
})

# 3. Get counts and percentages
cap_impact = pd.DataFrame({
    'posting_count': status.value_counts(),
    'percentage_of_total': (status.value_counts(normalize=True) * 100).round(2)
}).reset_index().rename(columns={'index': 'vacancy_status'})

display(cap_impact)

# Results: "The 'Cap at 20' rule would affect only 0.77% of the data 

# =================================================================
# 3. DATA ENGINEERING: THE DATA PIPELINE
# =================================================================
# This section implements the fixes for all issues identified above.

# 3.1 Keep your agency list 
agencies = '|'.join([
    'RECRUIT', 'MANPOWER', 'PERSONNEL', 'TALENT', 'ADVISORY', 
    'CONSULTANT', 'SEARCH', 'AGENCY', 'EMPLOYMENT', 'CAREER', 
    'SERVICES', 'STAFFING', 'JOBS', 'RESOURCE', 'CONSULTING', 
    'SOLUTIONS', 'HR', 'HUMAN RESOURCE', 'HEADHUNTER',
    'ANRADUS', 'PERSOLKELLY', 'RANDSTAD', 'ADECCO', 'FLINTEX', 
    'ZENITH', 'MTC', 'HYPERSCAL', 'MICHAEL PAGE', 'SCIENTEC',
    'MORGAN MCKINLEY', 'AMBITION GROUP', 'GOOD JOB CREATIONS', 
    'GMP TECHNOLOGIES', 'BEATHCHAPMAN', 'PEOPLE PROFILERS',
    'ROBERT HALF', 'PROSTAFF', 'EAMES', 'PASONA', 'WECRUIT', 
    'STAFFKING', 'ACCEO', 'ELITEZ', 'DYNAMIC HUMAN CAPITAL', 
    'PERSOLKELLY',
])

# 3.2 Consolidated Engineering Pipeline

# # SQL version for pipeline
# data_pipeline = f"""
# CREATE OR REPLACE TABLE potential_sg_jobs AS
# SELECT 
#     metadata_newPostingDate,
#     metadata_expiryDate, 
#     regexp_extract(categories, '"category":"([^"]+)"', 1) as clean_category,
#     -- 1. Company Consolidation (Applying findings from Section 1 (Company Normalization))
#     TRIM(TRAILING '.' FROM (
#         TRIM(REGEXP_REPLACE(REGEXP_REPLACE(UPPER(postedCompany_name), '\\(.*?\\)', '', 'g'), 
#              ' PTE| LTD| BANK| LIMITED| SINGAPORE| PRIVATE| BRANCH', '', 'g'))
#     )) as clean_company,

#     -- 2. Title Normalization (Simple version, Streamlit handles the rest)
#     LOWER(TRIM(
#     REGEXP_REPLACE(
#         REGEXP_REPLACE(
#             REGEXP_REPLACE(title, '\t', ' ', 'g'),  -- 1. Remove Tabs
#             ' \\|.*', '', 'g'                       -- 2. Cut after Pipe
#         ), 
#         '^[^a-zA-Z0-9]+|[^a-zA-Z0-9 ]+$', '', 'g'   -- 3. Remove leading/trailing symbols & emojis
#                 )
#         )) AS clean_title,

#     -- 3. Salary & Tiering (Applying findings from Section 2.2 (Realistic Salary Boundaries))
#     average_salary as monthly_pay,

#     -- 4. Job Status Filter (Applying findings from Section 2.4 (Bulk Clean with Status Filter))
#     status_jobStatus as job_status,

#     -- 5. Vacancy & Revenue Logic (New Inclusion)
#     -- We use COALESCE to treat NULL vacancies as 1, and LEAST to cap at 20 for realism
#     LEAST(COALESCE(numberOfVacancies, 1), 20) as adj_vacancies,

#     -- Calculating potential revenue directly in the pipeline (Monthly Pay * 12 * 20% * vacancies)
#     -- (average_salary * 12 * 0.20 * LEAST(COALESCE(numberOfVacancies, 1), 20)) as est_commission not used, as assume 1 post 1 lead even 20 vacancies
#     (average_salary * 12 * 0.20) * LEAST(numberOfVacancies, 1) as est_commission,

#     CASE 
#         WHEN average_salary >= 15000 OR positionLevels LIKE '%Senior Management%' THEN 'C-Suite'
#         WHEN average_salary >= 9000 OR positionLevels LIKE '%Manager%' THEN 'Senior / Managerial'
#         WHEN average_salary >= 5000 OR positionLevels LIKE '%Executive%' THEN 'Mid-Level'
#         ELSE 'Junior / Entry'
#     END as business_tier
# FROM 'SGJobData.csv'

# -- Rule from 2.4: Minimum Viable Data
# WHERE title IS NOT NULL 
#   AND postedCompany_name IS NOT NULL

# -- NEW: Safety check to remove rows that became blank after symbol stripping
#   AND clean_title != '' 
#   AND LENGTH(clean_title) >= 3 -- Ensures we don't have "junk" titles like "AA" or ".."
#   AND clean_title NOT SIMILAR TO '[0-9].*'
  
# -- Rule from 2.2.2 & 2.3.1: The "Cleanliness Wall"
# -- Removes $1 placeholders and $1.2M data entry errors
#   AND average_salary BETWEEN 1000 AND 50000

#    -- 4. Status Filter: Only include active hiring leads
#   AND LOWER(status_jobStatus) IN ('open', 're-open')
  
#   -- 5. Integrated Competitor Filter
#   -- AND NOT REGEXP_MATCHES(postedCompany_name, '{agencies}', 'i')
# """

# # Execute the consolidated logic
# # 1. First, execute the string to actually CREATE the table
# print("Final Data Profile Saved!")
# con.execute(data_pipeline)

# # 2. Now that the table 'gold_sg_jobs' exists, you can display it
# display(con.execute("SELECT * FROM potential_sg_jobs LIMIT 5").df())

# # SUMMARIZE of data_pipeline
# print("Final Data Profile Summary")
# display(con.execute("SUMMARIZE SELECT * FROM potential_sg_jobs").df())


# pandas version for pipeline
# 1. Start with filtering Min variable Data & Realistic Boundaries
pipeline_df = df[
    (df['title'].notna()) &
    (df['postedCompany_name'].notna()) &
    (df['average_salary'].between(1000, 50000)) &
    (df['status_jobStatus'].str.lower().isin(['open','re-open']))
].copy().rename(columns={
    'average_salary': 'monthly_pay',
    'status_jobStatus': 'job_status'
})    #avoid SettingWithCopyWarning

# 2. Company Normalization (The Regex parts)
pipeline_df['clean_company']=(
    pipeline_df['postedCompany_name']
    .str.upper()
    .str.replace(r'\(.*?\)','',regex=True) #Remove anything in brackets
    .str.replace(r' PTE| LTD| BANK| LIMITED| SINGAPORE| PRIVATE| BRANCH', '', regex=True)
    .str.strip()
    .str.strip('.')
)

# 3. Title & Category extraction
pipeline_df['clean_title'] = (
    pipeline_df['title']
    .str.replace(r'\t', ' ', regex=True)          # 1. Remove Tabs
    .str.split('|').str[0]                        # 2. Cut after Pipe
    .str.lower()
    .str.replace(r'^[^a-zA-Z0-9]+|[^a-zA-Z0-9 ]+$', '', regex=True) # 3. Symbol Stripper (!! and Emojis)
    .str.strip()
)

pipeline_df['clean_category'] = pipeline_df['categories'].str.extract(r'"category":"([^"]+)"')

# 4. Vacancy & Commission Logic
# Treat NULL as 1, cap at 20 for vacancies
pipeline_df['adj_vacancies'] = pipeline_df['numberOfVacancies'].fillna(1).clip(upper=20)
# Commission logic (assuming at least 1 vacancy for math)
COMMISSION_RATE = 0.2
pipeline_df['est_commission'] = (pipeline_df['monthly_pay'] * 12 * COMMISSION_RATE) * pipeline_df['numberOfVacancies'].fillna(1).clip(upper=1)

# 5. Business Tiering (THE CASE WHEN replacement)
# def tier_logic(row):
#     salary = row['average_salary']
#     level = str(row['positionLevels'])   # Adding str() prevents the code from crashing if a row is empty
#     if salary >= 15000 or 'Senior Management' in level:
#         return 'C-Suite'
#     elif salary >= 9000 or 'Manager' in level:
#         return 'Senior / Managerial'
#     elif salary >= 5000 or 'Executive' in level:
#         return 'Mid-Level'
#     else:
#         return 'Junior / Entry'
    
# pipeline_df['business_tier'] = pipeline_df.apply(tier_logic,axis = 1)

conditions = [
    (pipeline_df['monthly_pay'] >= 15000) | (pipeline_df['positionLevels'].str.contains('Senior Management', na=False)),
    (pipeline_df['monthly_pay'] >= 9000) | (pipeline_df['positionLevels'].str.contains('Manager', na=False)),
    (pipeline_df['monthly_pay'] >= 5000) | (pipeline_df['positionLevels'].str.contains('Executive', na=False))
]

choices = ['C-Suite', 'Senior / Managerial', 'Mid-Level']
pipeline_df['business_tier'] = np.select(conditions, choices, default='Junior / Entry' )

# display(pipeline_df.head())

# 6. Final Selection (Keep only the pipeline columns)
pipeline_columns = [
    'metadata_newPostingDate', 'metadata_expiryDate', 'clean_category', 
    'clean_company', 'clean_title', 'monthly_pay', 
    'job_status', 'adj_vacancies', 'est_commission', 'business_tier'
]
pipeline_df = pipeline_df[pipeline_columns]

# 7. Final Quality Control (The "Safety Filters")
# This replicates your SQL WHERE clause fixes
pipeline_df = pipeline_df[
    (pipeline_df['clean_title'] != '') &          # Remove blanks created by stripping
    (pipeline_df['clean_title'].str.len() >= 3) & # Remove very short junk
    (~pipeline_df['clean_title'].str.contains(r'^[0-9]', regex=True)) # Remove titles starting with numbers
].copy()

display(pipeline_df.head())

print("Final Data Profile Summary")
display(get_summary(pipeline_df))

# Save the Pandas version
print("Final Data Profile Saved!")
con.execute("CREATE OR REPLACE TABLE potential_sg_jobs_pd AS SELECT * FROM pipeline_df")

con.close()

 







Initial Data Profile Summary


,column_type,nulls,unique,min,max
categories,object,3988,21125,"[{""id"":1,""category"":""Accounting / Auditing / T...",nan
employmentTypes,object,3988,8,Contract,nan
metadata_expiryDate,object,3988,453,2023-04-04,nan
metadata_isPostedOnBehalf,bool,0,2,False,True
metadata_jobPostId,object,3988,1044597,ATS-2023-0275189,nan
metadata_newPostingDate,object,3988,431,2023-02-24,nan
metadata_originalPostingDate,object,3988,603,2022-10-03,nan
metadata_repostCount,int64,0,3,0,2
metadata_totalNumberJobApplication,int64,0,376,0,1342
metadata_totalNumberOfView,int64,0,1552,0,8190


average_salary         0
status_jobStatus    3988
dtype: int64

Zero Salaries: 3988

Salary Outliers:


,categories,employmentTypes,metadata_expiryDate,metadata_isPostedOnBehalf,metadata_jobPostId,metadata_newPostingDate,metadata_originalPostingDate,metadata_repostCount,metadata_totalNumberJobApplication,metadata_totalNumberOfView,...,occupationId,positionLevels,postedCompany_name,salary_maximum,salary_minimum,salary_type,status_id,status_jobStatus,title,average_salary
92072,"[{""id"":27,""category"":""Medical / Therapy Servic...",Full Time,2023-06-08,False,MCF-2023-0165468,2023-05-09,2023-03-02,2,51,764,...,NaN,Non-executive,HILL GROVE MEDICAL PTE. LTD.,10000000,1500,Monthly,0,Closed,Clinic assistant,5000750.0
359527,"[{""id"":1,""category"":""Accounting / Auditing / T...",Full Time,2023-09-27,False,MCF-2023-0659775,2023-08-28,2023-08-28,0,0,1,...,NaN,Executive,RK RECRUITMENT PTE. LTD.,25330000,2800,Monthly,0,Open,Accounts Executive - GT,12666400.0
1048575,"[{""id"":1,""category"":""Accounting / Auditing / T...",Internship/Attachment,2024-12-12,True,RANDOM_JOB_20251115011346015685_0,2024-01-16,2024-01-12,0,921,1604,...,NaN,Middle Management,BOND CAPITAL GROUP PTE. LTD.,6142101,107908,Monthly,0,Closed,Senior Manager - Operations,3125004.5
1048576,"[{""id"":8,""category"":""Customer Service""},{""id"":...",Internship/Attachment,2023-10-03,False,RANDOM_JOB_20251115011346673349_1,2023-03-09,2023-03-02,0,156,2559,...,NaN,Non-executive,ASCEND INTERNATIONAL TRAINING PTE. LTD.,10734314,108872,Monthly,0,Closed,Resident Physician,5421593.0
1048577,"[{""id"":2,""category"":""Admin / Secretarial""},{""i...",Part Time,2024-06-18,True,RANDOM_JOB_20251115011347120191_2,2023-12-05,2023-12-05,0,626,192,...,NaN,Manager,DYNAMIC HUMAN CAPITAL PTE. LTD.,2720804,276583,Monthly,0,Open,Senior Logistics Executive (1 yr contract) - u...,1498693.5
1048578,"[{""id"":13,""category"":""Environment / Health""},{...",Part Time,2024-11-25,True,RANDOM_JOB_20251115011347466118_3,2024-01-30,2024-01-17,0,1199,1754,...,NaN,Middle Management,ASCOTT INTERNATIONAL MANAGEMENT PTE LTD,20862169,324072,Monthly,0,Re-open,Language Teacher,10593120.5
1048579,"[{""id"":7,""category"":""Consulting""},{""id"":8,""cat...",Contract,2024-05-11,False,RANDOM_JOB_20251115011347817248_4,2024-01-16,2023-12-30,1,914,8103,...,NaN,Professional,RK RECRUITMENT PTE. LTD.,15531134,262482,Monthly,0,Re-open,"Sales Associate (Home Audio, Retail)",7896808.0
1048580,"[{""id"":4,""category"":""Architecture / Interior D...",Part Time,2024-07-22,False,RANDOM_JOB_20251115011348190957_5,2023-10-06,2023-09-30,1,1111,2370,...,NaN,Senior Management,SAFRAN LANDING SYSTEMS SERVICES SINGAPORE PTE....,23712119,14719,Monthly,0,Re-open,Executive Secretary,11863419.0
1048581,"[{""id"":16,""category"":""General Management""},{""i...",Freelance,2024-03-18,False,RANDOM_JOB_20251115011348553903_6,2023-09-04,2023-08-12,1,131,1626,...,NaN,Executive,RECRUIT EXPRESS PTE LTD,7859259,267303,Monthly,0,Closed,Junior Project Manager (IT Infrastructure),4063281.0
1048582,"[{""id"":2,""category"":""Admin / Secretarial""},{""i...",Contract,2023-08-16,False,RANDOM_JOB_20251115011348901570_7,2023-02-24,2023-01-26,1,580,3912,...,NaN,Executive,MINDFLEX EDUCATION PTE. LTD.,13798518,260117,Monthly,0,Re-open,Social media content creator,7029317.5


...

,vacancy_status,posting_count,percentage_of_total
0,At or Below Cap (Original Data),1036569,99.23
1,Above Cap (Clipped to 20),8028,0.77


,metadata_newPostingDate,metadata_expiryDate,clean_category,clean_company,clean_title,monthly_pay,job_status,adj_vacancies,est_commission,business_tier
84,2023-06-26,2023-07-26,Engineering,TIANCHENG ELECTRICAL MARINE EQUIPMENT,qa engineer,2650.0,Re-open,2,6360.0,Mid-Level
148,2023-06-09,2023-07-09,Admin / Secretarial,KEYSTONE CLINIC & SURGERY,clinic manager,2850.0,Re-open,1,6840.0,Senior / Managerial
169,2023-06-10,2023-07-10,Admin / Secretarial,EVERSAFE ACADEMY,centre in-charge,2750.0,Re-open,2,6600.0,Mid-Level
186,2023-06-21,2023-07-21,Consulting,QUANTUM LEAP CAREER CONSULTANCY,luxury client care services,3100.0,Re-open,2,7440.0,Mid-Level
221,2023-06-22,2023-07-22,Education and Training,SUPERGENIUS PRESCHOOL HBB,preschool speech & drama teacher,3200.0,Re-open,2,7680.0,Junior / Entry


Final Data Profile Summary


,column_type,nulls,unique,min,max
metadata_newPostingDate,object,0,365,2023-05-31,2024-05-29
metadata_expiryDate,object,0,365,2023-06-30,2024-06-28
clean_category,object,0,43,Accounting / Auditing / Taxation,Wholesale Trade
clean_company,object,0,49380,"""K"" LINE",ZZV VENTURE
clean_title,object,0,288822,a & a manager,zumba instructor
monthly_pay,float64,0,3493,1000.0,50000.0
job_status,object,0,2,Open,Re-open
adj_vacancies,int64,0,20,1,20
est_commission,float64,0,3493,2400.0,120000.0
business_tier,object,0,4,C-Suite,Senior / Managerial


Final Data Profile Saved!
